# Bước 2: Retrieval và Đánh giá (Faiss & Sentence-Transformers)
Notebook này thực hiện việc nhúng (embedding) 3 loại Corpus (Passage, Sentence, Proposition) và 1000 câu hỏi. 
Sau đó dùng Faiss để tìm kiếm Top-100 và Map kết quả về Passage để tính Recall@5, Recall@20.

In [ ]:
!pip install sentence-transformers faiss-cpu pandas pyarrow huggingface_hub

## Nạp HF Token (Chống Rate Limit khi tải 4 Models)

In [ ]:
from huggingface_hub import login
import os

# ĐIỀN TOKEN CỦA BẠN VÀO ĐÂY
TOKEN_CUA_BAN = 'hf_xxxxxxxxx'
login(token=TOKEN_CUA_BAN)
os.environ["HF_TOKEN"] = TOKEN_CUA_BAN
print("Đã nạp Token thành công!")

## 1. Tải Dữ liệu

In [ ]:
import pandas as pd
import numpy as np
import json
import torch
import faiss
import gc
from sentence_transformers import SentenceTransformer, models

print('Đang tải dữ liệu...')
# Chú ý: Đường dẫn này phải trỏ vào folder Input mà bạn vừa Add Data từ Notebook 1
input_dir = '/kaggle/input/TÊN_THƯ_MỤC_NOTEBOOK_1_CỦA_BẠN'  # SỬA DÒNG NÀY

questions_df = pd.read_json(f'{input_dir}/test_questions.jsonl', lines=True)

corpus = {
    'passage': pd.read_parquet(f'{input_dir}/corpus_passage.parquet'),
    'sentence': pd.read_parquet(f'{input_dir}/corpus_sentence.parquet'),
    'proposition': pd.read_parquet(f'{input_dir}/corpus_proposition.parquet')
}

texts = {}
for level in corpus:
    texts[level] = corpus[level]['contents'].tolist()
    print(f"{level.capitalize()}: {len(texts[level])} dòng")

passage_dict = dict(zip(corpus['passage']['id'], corpus['passage']['contents']))

def extract_exact_title(row):
    span = row['metadata']['title_span']
    return row['contents'][span[0]:span[1]]

passage_title_dict = dict(zip(corpus['passage']['id'], corpus['passage'].apply(extract_exact_title, axis=1)))


## 2. Các hàm Đánh giá, Tìm kiếm và Load Model (Xử lý Pooling Cấp thấp)

In [ ]:
def load_rigorous_model(model_name_or_path, device='cpu'):
    """
    Đảm bảo tuyệt đối chiến lược Pooling khớp 100% với bài báo gốc.
    """
    if 'sentence-transformers' in model_name_or_path:
        return SentenceTransformer(model_name_or_path, device=device)
    
    word_embedding_model = models.Transformer(model_name_or_path)
    dim = word_embedding_model.get_word_embedding_dimension()
    
    if 'contriever' in model_name_or_path.lower():
        pooler = models.Pooling(dim, pooling_mode_mean_tokens=True, pooling_mode_cls_token=False, pooling_mode_max_tokens=False)
    elif 'simcse' in model_name_or_path.lower() or 'dpr' in model_name_or_path.lower():
        pooler = models.Pooling(dim, pooling_mode_mean_tokens=False, pooling_mode_cls_token=True, pooling_mode_max_tokens=False)
    else:
        pooler = models.Pooling(dim)
        
    return SentenceTransformer(modules=[word_embedding_model, pooler], device=device)

def has_answer(answers, text):
    text = str(text).lower()
    if not isinstance(answers, list):
        answers = [answers]
    for ans in answers:
        if str(ans).lower() in text:
            return True
    return False

def search_and_map(q_emb, c_emb, level, corpus_df, top_k=100):
    d = c_emb.shape[1]
    index = faiss.IndexFlatIP(d)
    index.add(c_emb)
    
    search_k = top_k if level == 'passage' else top_k * 10
    scores, indices = index.search(q_emb, search_k)
    
    results = []
    for i, (row_scores, row_indices) in enumerate(zip(scores, indices)):
        passage_scores = {}
        for score, idx in zip(row_scores, row_indices):
            unit_id = corpus_df.iloc[idx]['id']
            passage_id = unit_id if level == 'passage' else unit_id.rsplit('-', 1)[0]
            if passage_id not in passage_scores:
                passage_scores[passage_id] = score
        
        top_passages = sorted(passage_scores.items(), key=lambda x: x[1], reverse=True)[:top_k]
        results.append(top_passages)
        
    del index
    return results

## 3. Chạy Pipeline (Sử dụng 2 GPU Song Song)

In [ ]:
models_config = {
    'Contriever': 'facebook/contriever',
    'SimCSE': 'princeton-nlp/sup-simcse-bert-base-uncased',
    'GTR': 'sentence-transformers/gtr-t5-base',
    'DPR': ('sentence-transformers/facebook-dpr-question_encoder-multiset-base', 
            'sentence-transformers/facebook-dpr-ctx_encoder-multiset-base')
}

all_metrics = []
num_gpus = torch.cuda.device_count()
print(f"Phát hiện {num_gpus} GPU(s)!")

for model_name, paths in models_config.items():
    print(f"\n{'='*40}\nĐang xử lý Model: {model_name}\n{'='*40}")
    
    if model_name == 'DPR':
        q_model = load_rigorous_model(paths[0], device='cpu')
        c_model = load_rigorous_model(paths[1], device='cpu')
    else:
        q_model = load_rigorous_model(paths, device='cpu')
        c_model = q_model

    pool = c_model.start_multi_process_pool()
        
    print("Encoding questions...")
    questions_text = questions_df['question'].tolist()
    q_emb = q_model.encode(questions_text, batch_size=64, normalize_embeddings=True, show_progress_bar=False)
    # Ép kiểu dữ liệu C-contiguous float32 cho mảng truy vấn của Faiss
    q_emb = np.ascontiguousarray(q_emb, dtype=np.float32)
    
    for level in ['passage', 'sentence', 'proposition']:
        print(f"\n--> Đang nhúng Corpus: {level} ({len(texts[level])} dòng)")
        
        b_size = 64 if level == 'passage' else 128
        
        c_emb = c_model.encode(texts[level], pool=pool, batch_size=b_size)
        
        # FIX: Ép mảng numpy trở thành dạng C-contiguous float32 (Bắt buộc với Faiss)
        c_emb = np.ascontiguousarray(c_emb, dtype=np.float32)
        faiss.normalize_L2(c_emb)
        
        print("Đang tìm kiếm & Map về Passage...")
        results = search_and_map(q_emb, c_emb, level, corpus[level], top_k=100)
        
        print("Đang tính Recall...")
        recalls = {5: 0, 20: 0, 100: 0}
        gold_page_recalls = {5: 0, 20: 0}
        
        for i, top_passages in enumerate(results):
            ans_list = questions_df.iloc[i]['answers']
            gold_title = str(questions_df.iloc[i]['title']).lower().strip()
            
            for k in [5, 20, 100]:
                found_ans = False
                for p_id, _ in top_passages[:k]:
                    p_text = passage_dict.get(p_id, "")
                    if has_answer(ans_list, p_text):
                        found_ans = True
                        break
                if found_ans:
                    recalls[k] += 1
                    
                if k in gold_page_recalls:
                    found_title = False
                    for p_id, _ in top_passages[:k]:
                        p_title = passage_title_dict.get(p_id, "").lower().strip()
                        if gold_title in p_title or p_title in gold_title:
                            found_title = True
                            break
                    if found_title:
                        gold_page_recalls[k] += 1
                        
        n = len(questions_df)
        metrics = {
            'Model': model_name,
            'Granularity': level,
            'Recall@5': round(recalls[5] / n * 100, 2),
            'Recall@20': round(recalls[20] / n * 100, 2),
            'Recall@100': round(recalls[100] / n * 100, 2),
            'Gold_Page_Recall@5': round(gold_page_recalls[5] / n * 100, 2),
            'Gold_Page_Recall@20': round(gold_page_recalls[20] / n * 100, 2)
        }
        all_metrics.append(metrics)
        print(f"Kết quả: R@5={metrics['Recall@5']} | R@20={metrics['Recall@20']}")
        
        del c_emb
        gc.collect()

    c_model.stop_multi_process_pool(pool)
        
    if model_name == 'DPR':
        del c_model, q_model
    else:
        del q_model, c_model
    gc.collect()
    torch.cuda.empty_cache()


## 4. Tổng hợp và Tạo Báo cáo Markdown

In [ ]:
results_df = pd.DataFrame(all_metrics)
results_df.to_csv('/kaggle/working/table3_recall.csv', index=False)
display(results_df)

# Tạo file Markdown hiển thị kết quả (table3_recall.md)
md_content = "# Kết quả Table 3: Dense X Retrieval\n\n"
md_content += "Bảng dưới đây so sánh hiệu suất truy xuất (Passage Recall) của các Model theo 3 cấp độ (Granularity) khác nhau:\n\n"
md_content += results_df.to_markdown(index=False)

with open('/kaggle/working/table3_recall.md', 'w', encoding='utf-8') as f:
    f.write(md_content)

print("\nHOÀN TẤT BƯỚC 2! Đã lưu table3_recall.csv và table3_recall.md")
